# Support Vector Machine (SVM) Model

Train and evaluate a Support Vector Machine classifier, run hyperparameter tuning with GridSearchCV, and plot the confusion matrix.

In [ ]:
# Check if running in Google Colab and set up environment cleanly
import os
import shutil
from pathlib import Path

if 'COLAB_RELEASE_TAG' in os.environ:
    # 1. Force change directory back to the root '/content'
    os.chdir('/content')
    
    # 2. Clean up any accidental nested clone directories to save space and resolve path confusion
    nested_path = Path('/content/Bird-Species-Classification-ML/Bird-Species-Classification-ML')
    if nested_path.exists():
        print("Cleaning up accidental nested git clone folders...")
        shutil.rmtree(nested_path, ignore_errors=True)
        
    # 3. Clone repository if it doesn't exist under /content
    if not os.path.exists('Bird-Species-Classification-ML'):
        print("Cloning Bird-Species-Classification-ML repository...")
        !git clone https://github.com/01329582993/Bird-Species-Classification-ML.git
        
    # 4. Change working directory to '/content/Bird-Species-Classification-ML'
    %cd /content/Bird-Species-Classification-ML
else:
    print("Running locally. Working directory:", os.getcwd())

In [ ]:
# Load pre-computed features and labels
# If features are missing or outdated, automatically re-extract with improved feature set
import os
import sys
import shutil
import numpy as np
import joblib
from pathlib import Path
from sklearn.preprocessing import StandardScaler

DATA_DIR = Path("./processed_data")
DATA_DIR.mkdir(parents=True, exist_ok=True)
npz_file = DATA_DIR / "combined_hog_color_lbp.npz"

# ---- Version check: delete stale/old feature cache ----
# New combined features have >20000 dims (4xHOG + 9xColor + 2xLBP + spatial pyramid)
# Old features had only 8206 dims (1xHOG + RGB-only + single LBP)
if npz_file.exists():
    _check = np.load(npz_file)
    if _check['X_train'].shape[1] < 20000:
        print(f"[Version check] Old feature cache detected ({_check['X_train'].shape[1]} dims). Deleting and re-extracting improved features...")
        for _f in DATA_DIR.glob('*.npz'): _f.unlink()
        for _f in DATA_DIR.glob('*.npy'): _f.unlink()
    del _check

if not npz_file.exists():
    print("Feature files not found. Running automatic feature extraction pipeline with cropping...")
    import pandas as pd
    import kagglehub
    from tqdm.auto import tqdm
    from PIL import Image
    from skimage.feature import hog, local_binary_pattern
    import cv2
    
    # ---- Step 1: Download and prepare dataset ----
    def locate_metadata():
        for p in [Path("metadata_preprocessed.csv"), DATA_DIR / "metadata_preprocessed.csv"]:
            if p.exists():
                return p
        return None

    metadata_file = locate_metadata()
    dataset_preprocessed_dir = Path("dataset_20_species_preprocessed")

    if metadata_file is None or not dataset_preprocessed_dir.exists():
        print("  Downloading CUB-200-2011 dataset via Kaggle Hub...")
        download_path = Path(kagglehub.dataset_download("wenewone/cub2002011"))
        DATASET_ROOT = download_path / "CUB_200_2011"
        IMAGES_FOLDER = DATASET_ROOT / "images"
        
        SUBSET_DIR = Path("./dataset_20_species")
        if SUBSET_DIR.exists():
            shutil.rmtree(SUBSET_DIR)
        SUBSET_DIR.mkdir(parents=True, exist_ok=True)
        
        bd_keywords = [
            'Crow', 'Kingfisher', 'Hummingbird', 'Mallard', 'Warbler',
            'Towhee', 'Jay', 'Creeper', 'Waxwing', 'Cuckoo',
            'Thrush', 'Woodpecker', 'Wren', 'Vireo', 'Catbird',
            'Meadowlark', 'Blackbird', 'Gull', 'Tern', 'Pelican'
        ]
        species_folders = sorted([f for f in IMAGES_FOLDER.iterdir() if f.is_dir()])
        selected_species = []
        for folder in species_folders:
            if any(kw.lower() in folder.name.lower() for kw in bd_keywords):
                if folder not in selected_species:
                    selected_species.append(folder)
            if len(selected_species) == 20:
                break
        
        for species_path in selected_species:
            shutil.copytree(str(species_path), SUBSET_DIR / species_path.name)
        
        classes_df = pd.read_csv(DATASET_ROOT / "classes.txt", sep=r"\s+", names=["class_id", "class_name"])
        images_df = pd.read_csv(DATASET_ROOT / "images.txt", sep=r"\s+", names=["image_id", "image_path"])
        labels_df = pd.read_csv(DATASET_ROOT / "image_class_labels.txt", sep=r"\s+", names=["image_id", "class_id"])
        bboxes_df = pd.read_csv(DATASET_ROOT / "bounding_boxes.txt", sep=r"\s+", names=["image_id", "x", "y", "width", "height"])
        
        selected_names = [p.name for p in selected_species]
        dataset_info = images_df.merge(labels_df, on="image_id").merge(classes_df, on="class_id").merge(bboxes_df, on="image_id")
        dataset_info = dataset_info[dataset_info["class_name"].isin(selected_names)].copy().reset_index(drop=True)
        dataset_info["full_image_path"] = dataset_info["image_path"].apply(lambda p: SUBSET_DIR / p)
        
        sys.path.append(str(Path(".").resolve()))
        import importlib
        import src.preprocessing
        importlib.reload(src.preprocessing)
        from src.preprocessing import create_stratified_splits, preprocess_and_save_image
        
        split_metadata = create_stratified_splits(dataset_info, train_ratio=0.7, val_ratio=0.15, test_ratio=0.15, random_state=42)
        
        if dataset_preprocessed_dir.exists():
            shutil.rmtree(dataset_preprocessed_dir)
        dataset_preprocessed_dir.mkdir(parents=True, exist_ok=True)
        
        preprocessed_paths = []
        for idx, row in split_metadata.iterrows():
            dst_file = dataset_preprocessed_dir / Path(row["image_path"])
            bbox = (row["x"], row["y"], row["width"], row["height"])
            preprocess_and_save_image(row["full_image_path"], dst_file, target_size=(224, 224), bbox=bbox)
            preprocessed_paths.append(str(dst_file))
        split_metadata["preprocessed_image_path"] = preprocessed_paths
        cols = ["image_id", "image_path", "class_id", "class_name", "split", "preprocessed_image_path", "x", "y", "width", "height"]
        split_metadata[cols].to_csv("metadata_preprocessed.csv", index=False)
        split_metadata[cols].to_csv(DATA_DIR / "metadata_preprocessed.csv", index=False)
        metadata_file = Path("metadata_preprocessed.csv")
        print("  Dataset preparation complete!")

    # ---- Step 2: Improved Feature Extraction ----
    # Multi-channel HOG (grayscale + R + G + B channels)
    def _extract_hog(img):
        arr = np.array(img.resize((128, 128), Image.Resampling.LANCZOS))  # H x W x 3
        hog_params = dict(orientations=9, pixels_per_cell=(8,8), cells_per_block=(2,2), block_norm='L2-Hys', visualize=False)
        gray = np.mean(arr, axis=2).astype(np.uint8)
        feats = [hog(gray, **hog_params)]
        for ch in range(3):
            feats.append(hog(arr[:,:,ch], **hog_params))
        return np.concatenate(feats).astype(np.float32)

    # Spatial pyramid HOG: full image + 4 quadrants (5 regions total)
    def _extract_spatial_hog(img):
        img128 = img.resize((128, 128), Image.Resampling.LANCZOS)
        w, h = img128.size
        hw, hh = w // 2, h // 2
        regions = [
            img128,                                    # full
            img128.crop((0,  0,  hw, hh)),             # top-left
            img128.crop((hw, 0,  w,  hh)),             # top-right
            img128.crop((0,  hh, hw, h)),              # bottom-left
            img128.crop((hw, hh, w,  h))              # bottom-right
        ]
        feats = []
        for region in regions:
            region_arr = np.array(region.resize((64, 64), Image.Resampling.LANCZOS))
            gray = np.mean(region_arr, axis=2).astype(np.uint8)
            feats.append(hog(gray, orientations=9, pixels_per_cell=(8,8), cells_per_block=(2,2), block_norm='L2-Hys', visualize=False))
        return np.concatenate(feats).astype(np.float32)

    # Multi-colorspace histogram (RGB + HSV + LAB)
    def _extract_color(img):
        img_rgb = np.array(img.convert("RGB"), dtype=np.uint8)
        img_hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)
        img_lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)
        all_hists = []
        for cs_img, ranges in [(img_rgb, [(0,256)]*3), (img_hsv, [(0,180),(0,256),(0,256)]), (img_lab, [(0,256)]*3)]:
            for ch in range(3):
                h, _ = np.histogram(cs_img[:,:,ch], bins=32, range=ranges[ch])
                h = h.astype(np.float32); h /= (h.sum() + 1e-9)
                all_hists.append(h)
        return np.concatenate(all_hists)

    # Multi-scale LBP (radius 1 + radius 3)
    def _extract_lbp(img):
        img_arr = np.array(img.convert("L"), dtype=np.uint8)
        def _lbp_hist(arr, P, R):
            lbp = local_binary_pattern(arr, P=P, R=R, method="uniform")
            h, _ = np.histogram(lbp.ravel(), bins=np.arange(0, P+3), range=(0, P+2))
            h = h.astype(np.float32); h /= (h.sum() + 1e-9)
            return h
        return np.concatenate([_lbp_hist(img_arr, 8, 1), _lbp_hist(img_arr, 24, 3)])

    metadata_df = pd.read_csv(locate_metadata())
    X_hog, X_sp_hog, X_color, X_lbp, y_all, splits_all = [], [], [], [], [], []

    print("  Extracting multi-channel HOG + Spatial Pyramid + Color + LBP features...")
    for _, row in tqdm(metadata_df.iterrows(), total=len(metadata_df)):
        img_path = Path(row["preprocessed_image_path"])
        img = Image.open(img_path).convert("RGB")
        X_hog.append(_extract_hog(img))
        X_sp_hog.append(_extract_spatial_hog(img))
        X_color.append(_extract_color(img))
        X_lbp.append(_extract_lbp(img))
        y_all.append(row["class_id"] - 1)
        splits_all.append(row["split"])

    X_hog = np.array(X_hog, dtype=np.float32)
    X_sp_hog = np.array(X_sp_hog, dtype=np.float32)
    X_color = np.array(X_color, dtype=np.float32)
    X_lbp = np.array(X_lbp, dtype=np.float32)
    y_all = np.array(y_all, dtype=np.int32)
    splits_all = np.array(splits_all)
    X_all = np.concatenate([X_hog, X_sp_hog, X_color, X_lbp], axis=1)

    unique_classes = sorted(metadata_df["class_name"].unique())
    lm = {cls: idx for idx, cls in enumerate(unique_classes)}
    joblib.dump(lm, DATA_DIR / "label_mapping.pkl")

    def _save_split_npz(fname, X):
        np.savez_compressed(
            DATA_DIR / fname,
            X_train=X[splits_all=="train"], y_train=y_all[splits_all=="train"],
            X_val=X[splits_all=="val"],   y_val=y_all[splits_all=="val"],
            X_test=X[splits_all=="test"],  y_test=y_all[splits_all=="test"]
        )
    _save_split_npz("hog_features.npz", X_hog)
    _save_split_npz("color_features.npz", X_color)
    _save_split_npz("lbp_features.npz", X_lbp)
    _save_split_npz("combined_hog_color_lbp.npz", X_all)
    print(f"  Feature extraction complete! Combined dims: {X_all.shape[1]}")

# ---- Load the features ----
data = np.load(npz_file)
X_train, y_train = data["X_train"], data["y_train"]
X_val,   y_val   = data["X_val"],   data["y_val"]
X_test,  y_test  = data["X_test"],  data["y_test"]

# Standardize features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

label_mapping = joblib.load(DATA_DIR / "label_mapping.pkl")
class_names = [k.split('.')[-1].replace('_', ' ') for k in sorted(label_mapping, key=label_mapping.get)]

print(f"Loaded feature dataset: {npz_file.name}")
print(f"  Training set  : {X_train.shape} (Standardized)")
print(f"  Validation set: {X_val.shape} (Standardized)")
print(f"  Testing set   : {X_test.shape} (Standardized)")
print(f"  Classes ({len(class_names)}): {class_names[:5]}...")

## 1. Train Baseline SVM Classifier

In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("--- Training Baseline SVM (RBF kernel) ---")
# Features are already StandardScaler-normalized by the data loader
svm_baseline = SVC(kernel='rbf', C=10, random_state=42)
svm_baseline.fit(X_train, y_train)

y_pred_base = svm_baseline.predict(X_test)
baseline_acc = accuracy_score(y_test, y_pred_base)
print(f"Baseline SVM Test Accuracy: {baseline_acc * 100:.2f}%")

## 2. Hyperparameter Tuning with GridSearchCV

In [ ]:
from sklearn.model_selection import GridSearchCV

print("--- Hyperparameter Tuning with GridSearchCV ---")
param_grid = {
    'C': [1, 10, 50, 100],
    'kernel': ['rbf'],
    'gamma': ['scale', 'auto', 0.001, 0.01]
}

grid_search_svm = GridSearchCV(
    estimator=SVC(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_search_svm.fit(X_train, y_train)

print("\n--- Tuning Results ---")
print(f"Best Hyperparameters : {grid_search_svm.best_params_}")
print(f"Best Cross-Val Score : {grid_search_svm.best_score_ * 100:.2f}%")

## 3. Evaluate & Save Best SVM Model

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

best_svm = grid_search_svm.best_estimator_
y_pred = best_svm.predict(X_test)
final_acc = accuracy_score(y_test, y_pred)
print(f"Final Tuned SVM Test Accuracy: {final_acc * 100:.2f}%")

# Save model
MODELS_DIR = Path("./models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)
model_path = MODELS_DIR / "svm_model.pkl"
joblib.dump(best_svm, model_path)
print(f"Best SVM model saved to '{model_path}' successfully!")

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, target_names=class_names))

# Confusion Matrix
plt.figure(figsize=(10, 8))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix - Tuned SVM')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()